# 逐帧 RGB DCT-QIM：固定一次最小诊断

本 notebook 默认 Run all 执行已固定的一次物理包：同一个既有 DCT2 P0 latent，BASE/QIM 两臂、完整181帧；无新 DiT 生成、训练、R2 或参数扫描。源码来自研究分支 dev/local-joint-state-payload-v1，也可填写普通源码目录。

**这是交付给用户运行的入口，发布准备时未执行实验 cells。没有预填实验结果。** 首次运行需要原三组输出中的 b2-m3-e01/P0 和 protocol.json。SOURCE 可改为同一既定输入的实际目录。已有 OUTPUT 时只读记录，不重新下载/setup/调用模型或重试失败。

固定读量：2臂×4层×181帧×5lane×2副本×63位＝912240槽；身份16个E41、2560局部位槽。物理上限为 D181一次，原生head46+46次，K/KT各4201次，H264 encode/probe/decode各2次；CPU FP64求解上限4000步/7200秒。失败、缺失、未决均保留。

已知位置诊断不等于盲同步或身份认证，软分数不是校准LLR/存在概率。DCT2、E41与CDC32原语义保留；本包不构造新时间族到CDC32的资格适配。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 输入与只读恢复
先看 OUTPUT 是否已有记录；新尝试只检查既定输入路径。源码与依赖准备在下一步。

In [ ]:
from pathlib import Path
import json

ARMS = ('BASE','QIM')

LAYERS = ('preclip','clip','rgb8','h264')

BUDGETS = dict(D181=1, baseline_head=46, head_replay=46, K=4201, KT=4201,
               E41=16, codec_encode=2, codec_probe=2, codec_decode=2, DiT=0, training=0)

def load(path):
    return json.loads(Path(path).read_text(), parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))

def plan():
    rows = {f'{a}/{l}':dict(kind='time_read', slots=181*5*2*63, status='NOT_STARTED') for a in ARMS for l in LAYERS}
    rows.update({f'{a}/identity/{s}/{phase}':dict(kind='identity_read', slots=160, status='NOT_STARTED')
                 for a in ARMS for s in (0,40) for phase in range(4)})
    return rows

def denominators():
    return dict(sources=1, arms=2, frames=181, time_layers=4, time_read_slots=912240,
                identity_views=16, identity_local_bit_slots=2560, paired_identity_slots=1280)

def has_existing_output(output):
    output = Path(output)
    return output.exists() and any(output.iterdir())

def recover(output):
    output = Path(output); errors = {}; saved = {}
    for name in ('attempt','state','counts','solver','solver_progress','native','evaluation','result'):
        try:
            value = load(output/(name+'.json'))
            if not isinstance(value,dict): raise ValueError('saved record must be an object')
            saved[name] = value
        except (OSError, ValueError) as e: errors[name] = f'{type(e).__name__}: {e}'
    rows = plan()
    state = saved.get('state')
    legal = {'NOT_STARTED','STARTED','RETURNED','FAILED','INTERRUPTED'}
    for key, row in rows.items():
        value = state.get(key) if isinstance(state,dict) else None
        if not isinstance(value,dict) or not isinstance(value.get('status'),str) or value['status'] not in legal:
            row.update(status='UNRESOLVED_RECEIPT',cause='missing or invalid saved row')
        else:
            row.update(value)
            row.update(kind=plan()[key]['kind'],slots=plan()[key]['slots'])
    for row in rows.values():
        if row['status'] == 'STARTED': row['status'] = 'UNRESOLVED_INTERRUPTED'
    return dict(status='READ_ONLY_EXISTING_ATTEMPT', physical_calls_this_invocation=0,
                denominators=denominators(), rows=rows, saved=saved, read_errors=errors)

def recovery_summary(snapshot):
    """Summarize saved receipts only; no raw arrays, scoring, truth or scientific imports."""
    def count(value, maximum):
        return type(value) is int and 0 <= value <= maximum
    def time_row_valid(row):
        status = row.get('status')
        if not isinstance(status,str) or status not in ('RETURNED','NOT_STARTED','FAILED','INTERRUPTED'): return False
        if row.get('fixed_slots') != 114030: return False
        if status != 'RETURNED': return True
        fields = ('active_slots','observed_active_slots','bit_errors','erased_active_slots')
        if any(not count(row.get(k),114030) for k in fields): return False
        return (row['bit_errors'] <= row['observed_active_slots'] <= row['active_slots']
                and row['erased_active_slots']==row['active_slots']-row['observed_active_slots']
                and count(row.get('wrong_origin_candidate_count'),905))
    def identity_row_valid(row):
        required = ('start','phase','region','local_row','bit','value','status','common_complete_cell_eligible','group','correct')
        if any(k not in row for k in required): return False
        if (type(row['start']) is not int or row['start'] not in (0,40)
                or type(row['phase']) is not int or row['phase'] not in range(4)
                or row['region'] not in ('R0','R1')
                or type(row['local_row']) is not int or row['local_row'] not in range(1,11)
                or type(row['bit']) is not int or row['bit'] not in range(8)): return False
        status = row['status']
        if not isinstance(status,str) or status not in ('AVAILABLE','NOT_STARTED','NOT_ENTERED','FAILED','INTERRUPTED','UNRESOLVED','UNSUPPORTED'): return False
        available = status=='AVAILABLE'
        if (available and not count(row['value'],1)) or (not available and row['value'] is not None): return False
        full = row['common_complete_cell_eligible']
        if type(full) is not bool: return False
        if (full and not count(row['group'],3)) or (not full and row['group'] is not None): return False
        return type(row['correct']) is bool if full and available else row['correct'] is None
    def comparison_valid(row):
        if not isinstance(row,dict) or row.get('fixed_paired_slots')!=1280: return False
        if any(not count(row.get(k),1280) for k in ('common_complete_observed_slots','added_support','lost_support')): return False
        common = row['common_complete_observed_slots']
        for arm in ('base','qim'):
            errors, ber = row.get(arm+'_common_errors'), row.get(arm+'_common_ber')
            if not count(errors,common): return False
            if common==0:
                if ber is not None: return False
            elif type(ber) not in (int,float) or not 0 <= ber <= 1 or ber != errors/common: return False
        return True
    saved = snapshot.get('saved',{})
    result = saved.get('result',{}); evaluation = saved.get('evaluation',{})
    rows = snapshot.get('rows',{})
    counts = saved.get('counts',{})
    calls = {}
    for kind, planned in BUDGETS.items():
        row = counts.get(kind,{})
        started = row.get('started') if isinstance(row,dict) else None
        returned = row.get('returned') if isinstance(row,dict) else None
        valid = type(started) is int and type(returned) is int and 0 <= returned <= started <= planned
        calls[kind] = dict(planned=planned,started=started if valid else None,
            returned=returned if valid else None,status='RECORDED' if valid else 'UNRESOLVED_RECEIPT')
    state_counts = {}
    for row in rows.values():
        status = row.get('status','UNRESOLVED_RECEIPT')
        state_counts[status] = state_counts.get(status,0)+1
    layer_rows = evaluation.get('time',[])
    if not isinstance(layer_rows,list): layer_rows = []
    layers = []
    for arm in ARMS:
        for layer in LAYERS:
            key = f'{arm}/{layer}'
            matches = [x for x in layer_rows if isinstance(x,dict) and x.get('key')==key]
            candidate = matches[0] if len(matches)==1 else {}
            row = candidate if time_row_valid(candidate) else {}
            layers.append(dict(key=key,fixed_slots=181*5*2*63,read_status=rows.get(key,{}).get('status','UNRESOLVED_RECEIPT'),
                evaluation_status=row.get('status','POST_UNRESOLVED'),
                **{field:row.get(field) if row.get('status')=='RETURNED' else None for field in ('active_slots','observed_active_slots','bit_errors','erased_active_slots','wrong_origin_candidate_count')}))
    identity_rows = evaluation.get('identity',[])
    if not isinstance(identity_rows,list): identity_rows = []
    identity = []
    for arm in ARMS:
        selected = [x for x in identity_rows if isinstance(x,dict) and x.get('arm')==arm]
        complete = len(selected)==1280 and all(identity_row_valid(x) for x in selected)
        if complete:
            coordinates = {(x['start'],x['phase'],x['region'],x['local_row'],x['bit']) for x in selected}
            complete = len(coordinates)==1280
        known = [x for x in selected if x.get('status')=='AVAILABLE' and type(x.get('value')) is int and x['value'] in (0,1)]
        scored = sum(type(x.get('correct')) is bool for x in selected) if complete else None
        identity.append(dict(arm=arm,fixed_slots=1280,recorded_rows=len(selected),
            observed_bits=len(known) if complete else None,
            erased_or_unread_bits=1280-len(known) if complete else None,
            full_cell_scored_bits=scored,
            full_cell_errors=sum(x.get('correct') is False for x in selected) if scored else None,
            status='SAVED_ROWS' if complete else 'POST_UNRESOLVED'))
    comparison = evaluation.get('identity_comparison')
    comparison_ok = comparison_valid(comparison)
    unresolved = any(x in state_counts for x in ('UNRESOLVED_RECEIPT','UNRESOLVED_INTERRUPTED','STARTED'))
    final_status = result.get('status')
    status = ('UNRESOLVED_RECORDS' if unresolved or any(c['status']!='RECORDED' for c in calls.values())
              else final_status if final_status in ('COMPLETED_FIXED_RECORDS','STOPPED_FIXED_ATTEMPT') else 'UNRESOLVED_FINAL_RECORD')
    if status=='COMPLETED_FIXED_RECORDS' and any(row.get('status')!='RETURNED' for row in rows.values()):
        status='UNRESOLVED_RECORDS'
    if status=='COMPLETED_FIXED_RECORDS' and (result.get('evaluation_status')!='POST_RECORDED'
            or evaluation.get('status')=='POST_UNRESOLVED'
            or any(row['evaluation_status']=='POST_UNRESOLVED' for row in layers)
            or any(row['status']=='POST_UNRESOLVED' for row in identity)
            or not comparison_ok):
        status='UNRESOLVED_POST_RECORD'
    solver = saved.get('solver',saved.get('solver_progress',{}))
    metrics = solver.get('metrics',{})
    if not isinstance(metrics,dict): metrics = {}
    budget = metrics.get('budget',{})
    if not isinstance(budget,dict): budget = {}
    solver_summary = {k:solver.get(k) for k in ('status','reason','iterations','estimated_bytes')}
    solver_summary['metrics'] = {k:metrics.get(k) for k in ('split','target','a_stationarity','r_stationarity')}
    solver_summary['budget'] = {k:budget.get(k) for k in ('linf','outside','excess')}
    return dict(status=status,saved_result_status=final_status,stop=result.get('stop'),
        evaluation_status=result.get('evaluation_status','POST_UNRESOLVED'),denominators=denominators(),
        state_counts=state_counts,calls=calls,
        solver=solver_summary,
        native_status=saved.get('native',{}).get('status','UNRESOLVED'),time_layers=layers,identity=identity,
        identity_comparison=comparison if comparison_ok else None,read_errors=snapshot.get('read_errors',{}),
        physical_calls_this_invocation=0,claim_ceiling='saved-record display only; no media/identity/synchronization/quality scientific PASS')

import os, stat, subprocess, sys, tempfile, time, urllib.request, zipfile
SOURCE = Path('/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/B-Framewise-QIM-V1/fixed-e01-181-v1')
LOG_DIRECTORY = OUTPUT.with_name(OUTPUT.name+'-logs')
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_DIRECTORY = None  # Optional ordinary editable source folder.
SCRATCH = Path('/content/video-wm-framewise-qim-v1')
CAN_START = False
PREPARATION_ERROR = None
SOURCE_ROOT = None
try:
    EXISTING = has_existing_output(OUTPUT)
    if EXISTING:
        print('已有输出：只读恢复；跳过下载、setup、模型和科学调用。')
    else:
        needed = [SOURCE/'private/b2-m3-e01/P0_terminal.pt', SOURCE/'protocol.json']
        missing = [str(path) for path in needed if not path.is_file()]
        if missing:
            PREPARATION_ERROR = 'INPUT_MISSING: ' + ', '.join(missing)
            print(PREPARATION_ERROR)
            print('未启动。SOURCE 可指向同一既定运行的正确目录；不生成或替换输入。')
        else:
            CAN_START = True
except (OSError, ValueError) as error:
    PREPARATION_ERROR = f'{type(error).__name__}: {error}'
    print('输出/输入读取未决，未启动：', PREPARATION_ERROR)
print('固定分母：', denominators())


## 加载可用源码
已有尝试跳过本步骤。下载普通源码或使用可编辑目录，不校验摘要或固定版本。

In [ ]:

if CAN_START:
    try:
        SCRATCH.mkdir(parents=True, exist_ok=True)
        LOG_DIRECTORY.mkdir(parents=True, exist_ok=True)
        WORKSPACE = Path(tempfile.mkdtemp(prefix='run-', dir=SCRATCH))
        if SOURCE_DIRECTORY is None:
            url = 'https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/' + SOURCE_REF
            archive = WORKSPACE/'source.zip'
            urllib.request.urlretrieve(url, archive)
            destination = (WORKSPACE/'source').resolve()
            destination.mkdir()
            with zipfile.ZipFile(archive) as source:
                for member in source.infolist():
                    name = member.filename.replace('\\','/')
                    target = (destination/name).resolve()
                    if name.startswith('/') or any(':' in part for part in name.split('/')) or not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                        raise ValueError('unsafe source archive path: '+member.filename)
                source.extractall(destination)
            entries = list(destination.glob('*/experiments/wan_state_clock/b_framewise_qim_v1_run.py'))
            if len(entries) != 1:
                raise FileNotFoundError('available branch source lacks the QIM runner: '+url)
            SOURCE_ROOT = entries[0].parents[2]
        else:
            SOURCE_ROOT = Path(SOURCE_DIRECTORY).expanduser().resolve()
        CONFIG = SOURCE_ROOT/'experiments/wan_state_clock/configs/b_framewise_qim_v1.json'
        for path in (CONFIG, SOURCE_ROOT/'experiments/wan_state_clock/b_framewise_qim_v1_run.py', SOURCE_ROOT/'runtime/wan/b_framewise_qim_v1_backend.py'):
            if not path.is_file(): raise FileNotFoundError('usable QIM source missing: '+str(path))
        print('使用源码：', SOURCE_ROOT)
    except Exception as error:
        CAN_START = False
        PREPARATION_ERROR = f'{type(error).__name__}: {error}'
        print('源码准备失败，未启动：', PREPARATION_ERROR)


## 执行固定一次包
新尝试复用已跑通的依赖修复并启动独立子进程。日志放在 OUTPUT 同级的独立日志目录，避免提前占用 OUTPUT。中断后再次 Run all 仅读收据。

In [ ]:
if CAN_START:
    import time
    # Reuse the successfully run Card102 setup, including its torchao repair.
    # Keep this self-contained; no legacy notebook/builder runtime dependency.
    LOG = LOG_DIRECTORY/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, cwd=SOURCE_ROOT, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','-qq','update'], 'Prepare media dependency')
        command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))


    probe = 'import runtime.wan.b_framewise_qim_v1; import runtime.wan.b_framewise_qim_v1_backend; import experiments.wan_state_clock.b_framewise_qim_v1_run'
    command([sys.executable, '-c', probe], 'Probe actual QIM runner/backend imports')
    with (LOG_DIRECTORY/'runner.log').open('a') as stream:
        worker = subprocess.Popen([sys.executable, '-u', '-m',
            'experiments.wan_state_clock.b_framewise_qim_v1_run',
            '--source', str(SOURCE), '--output', str(OUTPUT), '--config', str(CONFIG)],
            cwd=SOURCE_ROOT, stdout=stream, stderr=subprocess.STDOUT)
    try:
        while worker.poll() is None:
            time.sleep(30)
            try:
                snapshot = recovery_summary(recover(OUTPUT))
                print('记录状态:', snapshot['state_counts'], '求解:', snapshot['solver'].get('status'),
                      '迭代:', snapshot['solver'].get('iterations'), flush=True)
            except (OSError, ValueError) as error:
                print('当前记录读取未决:', str(error), flush=True)
    except KeyboardInterrupt:
        worker.terminate()
        try: worker.wait(timeout=30)
        except subprocess.TimeoutExpired: worker.kill(); worker.wait()
        print('子进程已停止；已有尝试保持消耗，下次 Run all 只读。')
        print('恢复摘要:', recovery_summary(recover(OUTPUT)))
        raise
    print('worker exit:', worker.returncode, '日志:', LOG_DIRECTORY/'runner.log')


## 查看保存结果
以下展示求解、分层读量与身份记录，不运行解码器/评分器，不把未知补成零。完整逐项证据保留在 OUTPUT。

In [ ]:

if has_existing_output(OUTPUT):
    summary = recovery_summary(recover(OUTPUT))
    print('恢复状态:', summary['status'], '保存的最终状态:', summary['saved_result_status'])
    print('停止原因:', summary['stop'], '后评:', summary['evaluation_status'])
    print('固定分母:', summary['denominators'], '逐项状态:', summary['state_counts'])
    print('调用记录（None = 未知）:')
    for name, row in summary['calls'].items(): print(name, row)
    print('求解:', summary['solver'], '原生物化:', summary['native_status'])
    print('逐层保存读量（不是重评分；None = 未知）:')
    for row in summary['time_layers']: print(row)
    print('身份支持/错误/擦除:')
    for row in summary['identity']: print(row)
    print('共同完整cell比较:', summary['identity_comparison'])
    for name, error in summary['read_errors'].items(): print('记录不可用:', name, error)
else:
    print('没有尝试记录。准备状态:', PREPARATION_ERROR or '尚未进入运行cell')
    print('固定分母:', denominators())
print('这里只显示已有记录。真实成本、媒体存留、身份保持、盲同步、存在性/null标定及画质结论没有由工程检查证明。')
print('输出目录:', OUTPUT)
